##  Shop the look page

In [0]:
%pip install /dbfs/FileStore/shared_uploads/dmat/pmgoogle-1.4-py3-none-any.whl
%pip install  /dbfs/FileStore/shared_uploads/dmat/dmat-0.1-py3-none-any.whl

dbutils.widgets.text(
    "report_run_date",      # Widget name
    "",                     # Default value
    "Report Run Date"       # Optional label
)

In [0]:
from dmat.helpers import *
from dmat import spark_helpers as dsh

from dateutil.relativedelta import relativedelta
import datetime

spark.conf.set("spark.sql.shuffle.partitions","1024")
spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")
#sc.hadoopConfiguration.set("mapreduce.fileoutputcommitter.marksuccessfuljobs", "false")

##############
## Imports & Inits
import pyspark
from pyspark import *
from pyspark.sql import *
from dateutil import tz
from dateutil.relativedelta import relativedelta
import datetime

from pmgoogle.sheets import GoogleSheet
import numpy as np
import pandas as pd
import time
import json
import pyspark.sql.functions as F

# get service account creds
#athena-master@athena-284716.iam.gserviceaccount.com
#service_account = json.loads(dbutils.secrets.get(scope="data_pa-dmat_dmat-secrets-scope", key="athena_sa"))['general']  

sc = spark._sc
helpers = sc._jvm.com.poshmark.spark.helpers
Config = helpers.Config
Redshift = helpers.Redshift
SaveMode = sc._jvm.org.apache.spark.sql.SaveMode
SaveMode_Append = SaveMode.Append
SaveMode_Overwrite = SaveMode.Overwrite
SaveMode_Ignore = SaveMode.Ignore
SaveMode_Error = SaveMode.ErrorIfExists

## Load default configs for helper libraries and register UDFs
Config.reloadConfigs(sc._jsc.sc())
Config.registerUDFs()


# Returns the current UTC time
def getUTC():
    return datetime.datetime.utcnow().replace(tzinfo=tz.gettz('UTC'))

# Returns the current time in 'US/Pacific'
def getPacificTime(date=False, month=False):
    now = getUTC().astimezone(tz.gettz('US/Pacific'))
    if date:
        now = now.date()
    if month:
        now = now.replace(day=1)

    return now



dsh.init(spark)
Redshift = sc._jvm.com.poshmark.spark.helpers.Redshift

def get_widget(widget_name, default="", _type=str):
    try:
        tmp =  _type(dbutils.widgets.get(widget_name))
        if tmp:
            return tmp
        else:
            return default
    except:
        return default

run_now = get_widget('run_now',default='false')
if (run_now.lower() == 'true'):
    force = True
else:
    force = False

send_day = getPacificTime().isoweekday()
start_of_month = True if getPacificTime().day == 1 else False

## Get today & yesterday
local_date = getPacificTime() 
today = local_date.strftime('%Y-%m-%d')
yesterday = (local_date - datetime.timedelta(days=1)).strftime('%Y-%m-%d')

##############
## get a value from a widget
report_run_date = dbutils.widgets.get("report_run_date")
process_end_date = get_widget('report_run_date', default=yesterday)
## last_90_days = (datetime.datetime.strptime(process_end_date, "%Y-%m-%d") - datetime.timedelta(days=90)).strftime('%Y-%m-%d') 
last_90_days = (datetime.datetime.strptime(process_end_date, "%Y-%m-%d") - datetime.timedelta(days=3)).strftime('%Y-%m-%d')
## last_90_days = (datetime.datetime.strptime(process_end_date, "%Y-%m-%d") - datetime.timedelta(days=2)).strftime('%Y-%m-%d')
process_start_date = last_90_days

## process_date = get_widget("process_date",default=yesterday)
## process_end_date = get_widget("process_end_date",default=yesterday)

# print(f"process_date: {process_date}, process_end_date: {process_end_date}")

# stage_s3_path = "s3://data-tmp-poshmark-production/dmat/project_highway/top_stats/v2/"

def to_jvm_list(py_list):
    l = len(py_list)
    jvm_list = sc._gateway.new_array(sc._jvm.java.lang.String, l)
    for i in range(l):
        jvm_list[i] = py_list[i]
    return jvm_list
print(process_start_date, process_end_date)


In [0]:

process_start_date = last_90_days
process_end_date = process_end_date

print(f"report_run_date: {report_run_date}, process_start_date: {process_start_date}, process_end_date: {process_end_date}")

In [0]:
from pyspark.sql import *
from pyspark import *
from pyspark.sql import DataFrame

sc = spark._sc
helpers = sc._jvm.com.poshmark.spark.helpers
Config = helpers.Config
Redshift = helpers.Redshift
Config.reloadConfigs(sc._jsc.sc())
Config.registerUDFs()

query = "select * from analytics_scratch.l365d_shopper_segment"
DataFrame(Redshift.getQueryDF(query), spark).createOrReplaceTempView("l365d_shopper_segment")
# query = "select * from athena_scratch.core__users_active_user_segments_daily"
# DataFrame(Redshift.getQueryDF(query), spark).createOrReplaceTempView("core__users_active_user_segments_daily")

from the look, you can go to anywhere else;  ---search pages


main_page = look, it could come from Feed/Closet, and others because use could go to other and click back to feed, the look page will sit there; 


In [0]:
queryDF = spark.sql(f""" 
select 
event_date, 
user_id, 
prev_page_name, 
main_page_name, 
event_id, 
shopper_signup_segment_v3_l06, 
app_type, 
case when main_page_name = 'look'  then 'L0'
when prev_page_name = 'look' then 'L1' 
end as fm_lvl,
case when main_page_name = 'look'  then 'L0'
when prev_page_name = 'look' and main_page_name = 'search_results'  then 'L1_SearchResult'
when prev_page_name = 'look' and main_page_name = 'cross_merch_listings'  then 'L1_CrossMerchListing'
when prev_page_name = 'look' and main_page_name = 'feed'  then 'L1_feed'
when prev_page_name = 'look' and main_page_name = 'closet'  then 'L1_closet'
else 'Others' end as fm_lvl1,
case when shopper_listing_first_booked_date = event_date then 'Y' else 'N' END  AS d1_ordr_yn, 
case when shopper_listing_first_booked_date between  event_date  and date_add(event_date, 7) then 'Y' else 'N' END  AS d7_ordr_yn, 
listing_id, -- order items  if d1_ordr_yn = 'y'
shopper_listing_first_booked_date, 
shopper_listing_first_gmv, 
shopper_listing_first_order_id--orders
from external_scratch.highway_traffic_enriched
where 1=1 and (main_page_name = 'look'  or prev_page_name = 'look' )
 and event_date between '{process_start_date}' and '{process_end_date}'
 and  actor_type = 'user'
 and shopper_listing_interaction_number = 1
"""
)
#val s3_path = "s3://data-tmp-poshmark-production/qiong/for_you_feed/visual_discovery/ql_visual_discovery_look_page_fm_base/"
#queryDF.write.format("parquet").mode("Overwrite").save(s3_path)

In [0]:
output_path = "s3://analytics-scratch-reviewed-poshmark-production/qiong/for_you_feed/visual_discovery/ql_visual_discovery_look_page_fm_base/"
queryDF.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem.write.partitionBy("booked_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem_df.write.format("parquet").mode("Overwrite").save(output_path)

## ## revert back to dynamic from static
## spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_visual_discovery_look_page_fm_base")

##print(process_start_date)
##print(process_end_date)
print(output_path)

### convert temporary bucket to final bucket
s3_path_old = "s3://data-tmp-poshmark-production/qiong/for_you_feed/visual_discovery/ql_visual_discovery_look_page_fm_base/"
s3_path = "s3://analytics-scratch-reviewed-poshmark-production/qiong/for_you_feed/visual_discovery/ql_visual_discovery_look_page_fm_base/"
df = spark.read.format("parquet").load(s3_path_old)
df.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)


In [0]:
#s3_path = "s3://data-tmp-poshmark-production/qiong/for_you_feed/visual_discovery/ql_visual_discovery_look_page_fm_base/"
s3_path = "s3://analytics-scratch-reviewed-poshmark-production/qiong/for_you_feed/visual_discovery/ql_visual_discovery_look_page_fm_base/"
df = spark.read.format("parquet").load(s3_path)
df.createOrReplaceTempView("ql_visual_discovery_look_page_fm_base")

In [0]:
%sql 
select
min(event_date), 
max(event_date)
from ql_visual_discovery_look_page_fm_base 
limit 5

In [0]:
%sql 
select 
event_date, 
fm_lvl, 
fm_lvl1,
count(distinct user_id) as user_cnt, 
count(distinct event_id) as discoveries, --should not be distinct listing_id, it would be based on each user 
count(distinct case when shopper_listing_first_order_id is not null then listing_id end) as oi,
count(distinct  shopper_listing_first_order_id ) as ordr, 
sum(  shopper_listing_first_gmv ) as gmv, 

count(distinct listing_id + user_id) as fm, -- should not be distinct listing_id, it would be based on each user 
/** OI**/
count(distinct case when d1_ordr_yn  = 'Y' then listing_id end) as d1_oi,
/** order**/
count(distinct case when d1_ordr_yn  = 'Y' then shopper_listing_first_order_id end) as d1_ordr,
/** gmv**/
sum(distinct case when d1_ordr_yn  = 'Y' then shopper_listing_first_gmv  end) as d1_gmv,

count(distinct case when d7_ordr_yn  = 'Y' then listing_id end) as d7_oi,
count(distinct case when d7_ordr_yn  = 'Y' then shopper_listing_first_order_id end) as d7_ordr,
sum(distinct case when d7_ordr_yn  = 'Y' then shopper_listing_first_gmv  end) as d7_gmv
from ql_visual_discovery_look_page_fm_base 
group by 1,2,3
order by 1 desc


### Daily Look CTR
[Athena Report for Page references]( https://docs.google.com/spreadsheets/d/1RdjFnHD0w5aV3LMdaj9Nz0IWsAtBZzOcNMwp5F4SYIc/edit?gid=1847036290#gid=1847036290)

[First Match Notebook(Run it before run below)](https://poshmark-prod.cloud.databricks.com/editor/notebooks/1880820226657917?o=3891659053752709)

In [0]:
#bucket = "s3://data-tmp-poshmark-production" ## 30 day bucket
bucket = "s3://analytics-scratch-reviewed-poshmark-production" ## 1yr bucket
team_name = "product_analytics"
project_name = "feed/visual_discovery/look"
data_name = "lookDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_visual_discovery_look_page")

In [0]:
%sql 
select min(event_date), max(event_date) from ql_visual_discovery_look_page


#### Daily Active users

In [0]:
from pyspark.sql import *
from pyspark import *
from pyspark.sql import DataFrame

sc = spark._sc
helpers = sc._jvm.com.poshmark.spark.helpers
Config = helpers.Config
Redshift = helpers.Redshift
Config.reloadConfigs(sc._jsc.sc())
Config.registerUDFs()

query = "select * from  analytics.dw_user_events_daily"
DataFrame(Redshift.getQueryDF(query), spark).createOrReplaceTempView("dw_user_events_daily")
# query = "select * from athena_scratch.core__users_active_user_segments_daily"
# DataFrame(Redshift.getQueryDF(query), spark).createOrReplaceTempView("core__users_active_user_segments_daily")

In [0]:
queryDF = spark.sql(f"""
SELECT
    DATE(dw_user_events_daily.event_date) AS event_date,
    COUNT(DISTINCT CASE
        WHEN dw_user_events_daily.is_active
             AND dw_user_events_daily.is_valid_user
        THEN dw_user_events_daily.user_id
        ELSE NULL
    END) AS is_active_user
FROM dw_user_events_daily AS dw_user_events_daily
LEFT JOIN s3_analytics.dw_users AS dw_users
    ON dw_user_events_daily.user_id = dw_users.user_id
WHERE dw_user_events_daily.event_date BETWEEN  '{process_start_date}' and '{process_end_date}'
  AND COALESCE(dw_user_events_daily.app, 'unknown') IN ('unknown', 'iphone', 'ipad', 'external', 'android', 'web')
  AND dw_users.is_valid_user = TRUE
  AND dw_users.home_domain = 'us'
  AND NOT COALESCE(
        (
          DATEDIFF(
            CASE WHEN dw_users.user_status = 'restricted' THEN dw_users.status_updated_at ELSE NULL END,
            COALESCE(dw_users.guest_joined_at, dw_users.joined_at)
          ) + 1 <= 30
        ),
        FALSE
      )
GROUP BY 1
ORDER BY 1
""")

In [0]:
output_path = "s3://analytics-scratch-reviewed-poshmark-production/qiong/for_you_feed/visual_discovery/ql_au/"
queryDF.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem.write.partitionBy("booked_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem_df.write.format("parquet").mode("Overwrite").save(output_path)

## ## revert back to dynamic from static
## spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_au")
##print(process_start_date)
##print(process_end_date)
print(output_path)

In [0]:
output_path = "s3://analytics-scratch-reviewed-poshmark-production/qiong/for_you_feed/visual_discovery/ql_au/"

## base_orderitem.write.partitionBy("booked_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem_df.write.format("parquet").mode("Overwrite").save(output_path)

## ## revert back to dynamic from static
## spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_au")
##print(process_start_date)
##print(process_end_date)
print(output_path)

In [0]:
%sql 
select * from ql_au
limit 5

### Entry Clicks

In [0]:
output_path = "s3://analytics-scratch-reviewed-poshmark-production/qiong/for_you_feed/visual_discovery/visual_discovery_feed_content_lvl2/"

#queryDF.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem.write.partitionBy("booked_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem_df.write.format("parquet").mode("Overwrite").save(output_path)

## ## revert back to dynamic from static
## spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("visual_discovery_feed_content_lvl2")

##print(process_start_date)
##print(process_end_date)
print(output_path)



In [0]:
from pyspark.sql import *
from pyspark import *
from pyspark.sql import DataFrame

sc = spark._sc
helpers = sc._jvm.com.poshmark.spark.helpers
Config = helpers.Config
Redshift = helpers.Redshift
Config.reloadConfigs(sc._jsc.sc())
Config.registerUDFs()

query = "select * from analytics_scratch.l365d_shopper_segment"
DataFrame(Redshift.getQueryDF(query), spark).createOrReplaceTempView("l365d_shopper_segment")
# query = "select * from athena_scratch.core__users_active_user_segments_daily"
# DataFrame(Redshift.getQueryDF(query), spark).createOrReplaceTempView("core__users_active_user_segments_daily")

In [0]:
%sql
with ctr as (
select 
event_date,   
pm_start_of_week(event_date) AS event_week,
DATE(date_trunc('month', event_date)) event_month,
COUNT(case when r.verb = 'view' AND r.dob_name = 'look' then r.user_Id else null end) as look_views, 
COUNT(case when r.verb = 'observe' AND r.on_name = 'look' then r.user_Id else null end) as look_client_impressions,
-- lookunit client impressions
COUNT(distinct case when r.verb = 'observe' AND r.on_name = 'look' then CONCAT(r.user_Id,f_pm_epoch_to_pacific(r.at),r.story_type,r.unit_position) else null end) as look_unit_impressions,
/** clicks**/
COUNT(case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit' 
        and content_type = 'post' then r.user_Id else null end) as look_clicks,
        
COUNT(CASE WHEN r.verb = 'click' AND r.on_name = 'look'  AND r.unit_position IS NOT NULL AND r.listing_id IS NOT NULL AND r.dob_name = 'feed_unit'  and content_type = 'post'  THEN r.user_Id ELSE NULL END) AS look_listing_clicks, --listing_ID IS NOT NULL

COUNT(case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit'  and content_type = 'post'  and r.location = 'content' then r.user_Id else null end) as look_content_clicks,
COUNT(case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit'  and content_type = 'post' and r.location = 'footer' then user_Id else null end) as look_footer_clicks,

/** clickers**/

COUNT(DISTINCT case when r.verb = 'view' AND r.dob_name = 'look' then r.user_Id else null end) as look_viewers,
COUNT(DISTINCT case when r.verb = 'observe' AND r.on_name = 'look' then r.user_Id else null end) as look_client_impressions_user,

COUNT(DISTINCT case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit' 
        and content_type = 'post' then r.user_Id else null end) as look_clickers,
COUNT(DISTINCT CASE WHEN r.verb = 'click' AND r.on_name = 'look'  AND r.unit_position IS NOT NULL AND r.listing_id IS NOT NULL AND r.dob_name = 'feed_unit'  and content_type = 'post'  THEN r.user_Id ELSE NULL END) AS look_listing_clicker, --listing_ID IS NOT NULL

COUNT(DISTINCT case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit'  and content_type = 'post'  and r.location = 'content' then r.user_Id else null end) as look_content_clicker,
COUNT(DISTINCT case when r.verb = 'click' AND r.on_name = 'look' AND r.story_type is not null AND r.unit_position is not null AND r.dob_name = 'feed_unit'  and content_type = 'post' and r.location = 'footer' then user_Id else null end) as look_footer_clickers
from ql_visual_discovery_look_page r
group by 1
)

,fm as (
select 
event_date, 
count(distinct event_id) as fm_discoveries, --should not be distinct listing_id, it would be based on each user 
count(distinct case when shopper_listing_first_order_id is not null then listing_id end) as fm_oi,
count(distinct  shopper_listing_first_order_id ) as fm_ordr, 
sum( shopper_listing_first_gmv ) as fm_gmv, 

/** D1 oi **/
count(distinct case when d1_ordr_yn  = 'Y' then listing_id end) as fm_d1_oi,
/** D1 order**/
count(distinct case when d1_ordr_yn  = 'Y' then shopper_listing_first_order_id end) as fm_d1_ordr,
/** D1 gmv**/
count(distinct case when d1_ordr_yn  = 'Y' then shopper_listing_first_gmv  end) as fm_d1_gmv,

/** D7**/
count(distinct case when d7_ordr_yn  = 'Y' then listing_id end) as fm_d7_oi,
count(distinct case when d7_ordr_yn  = 'Y' then shopper_listing_first_order_id end) as fm_d7_ordr,
count(distinct case when d7_ordr_yn  = 'Y' then shopper_listing_first_gmv  end) as fm_d7_gmv,

/** L0 only **/

count(distinct case when fm_lvl = 'L0' then event_id end) as L0_fm_discoveries, --should not be distinct listing_id, it would be based on each user 
count(distinct case when fm_lvl = 'L0' and shopper_listing_first_order_id is not null then listing_id end) as L0_fm_oi,
count(distinct case when fm_lvl = 'L0' then shopper_listing_first_order_id end) as L0_fm_ordr, 
sum( case when fm_lvl = 'L0' then shopper_listing_first_gmv end ) as L0_fm_gmv, 

count(distinct case when fm_lvl = 'L0' and d1_ordr_yn  = 'Y' then listing_id end) as L0_fm_d1_oi,
count(distinct case when fm_lvl = 'L0' and d1_ordr_yn  = 'Y' then shopper_listing_first_order_id end) as L0_fm_d1_ordr,
sum(distinct case when fm_lvl = 'L0' and d1_ordr_yn  = 'Y' then shopper_listing_first_gmv  end) as L0_fm_d1_gmv,


count(distinct case when fm_lvl = 'L0' and d7_ordr_yn  = 'Y' then listing_id end) as L0_fm_d7_oi,
count(distinct case when fm_lvl = 'L0' and d7_ordr_yn  = 'Y' then shopper_listing_first_order_id end) as L0_fm_d7_ordr,
sum(distinct case when fm_lvl = 'L0' and d7_ordr_yn  = 'Y' then shopper_listing_first_gmv  end) as L0_fm_d7_gmv, 


/** L1 only **/

count(distinct case when fm_lvl = 'L1' then event_id end) as L1_fm_discoveries, --should not be distinct listing_id, it would be based on each user 
count(distinct case when fm_lvl = 'L1' and shopper_listing_first_order_id is not null then listing_id end) as L1_fm_oi,
count(distinct case when fm_lvl = 'L1' then shopper_listing_first_order_id end) as L1_fm_ordr, 
sum( case when fm_lvl = 'L1' then shopper_listing_first_gmv end ) as L1_fm_gmv, 

count(distinct case when fm_lvl = 'L1' and d1_ordr_yn  = 'Y' then listing_id end) as L1_fm_d1_oi,
count(distinct case when fm_lvl = 'L1' and d1_ordr_yn  = 'Y' then shopper_listing_first_order_id end) as L1_fm_d1_ordr,
sum(distinct case when fm_lvl = 'L1' and d1_ordr_yn  = 'Y' then shopper_listing_first_gmv  end) as L1_fm_d1_gmv,


count(distinct case when fm_lvl = 'L1' and d7_ordr_yn  = 'Y' then listing_id end) as L1_fm_d7_oi,
count(distinct case when fm_lvl = 'L1' and d7_ordr_yn  = 'Y' then shopper_listing_first_order_id end) as L1_fm_d7_ordr,
sum(distinct case when fm_lvl = 'L1' and d7_ordr_yn  = 'Y' then shopper_listing_first_gmv  end) as L1_fm_d7_gmv

from ql_visual_discovery_look_page_fm_base
where fm_lvl = 'L0' or (fm_lvl = 'L1' and fm_lvl1 = 'L1_SearchResult')
group by 1

), 
entry_clicks as (

select event_date, 
count(case when verb = 'observe' then user_id else null end) as mu_client_impressions,
count(case when verb = 'click' then user_id else null end) as mu_clicks,
count(distinct case when verb = 'observe' then user_id else null end) as mu_client_impressions_users,
count(distinct case when verb = 'click' then user_id else null end) as mu_clicks_users
 from visual_discovery_feed_content_lvl2
group by 1
order by 1

)

select ctr.*, 
fm.*,  
ec.*, 
au.is_active_user as dau
from ctr 
left join fm on ctr.event_date  =  fm.event_date
left join ql_au au on ctr.event_date = au.event_date 
left join entry_clicks ec on ctr.event_date = ec.event_Date
where 1=1
order by 1


### Other Page Views and Clicks 

In [0]:
%sql 

select 

sum(search_and_search_results_views)search_and_search_results_views,
sum(community_closet_views)community_closet_views, 
sum(brand_views)brand_views, 
sum(cross_merch_grid_views) cross_merch_grid_views,

sum(search_and_search_results_clicks)search_and_search_results_clicks,
sum(community_closet_clicks)community_closet_clicks, 
sum(brand_clicks)brand_clicks, 
sum(cross_merch_grid_clicks) cross_merch_grid_clicks 

from external_scratch.user_page_engagement_trends_daily r
inner join s3_analytics.dw_users dw_users on r.user_id = dw_users.user_id
WHERE 1=1
    AND event_date between '2025-12-01' and '2025-12-14'
    AND ((app_type IN ('iphone', 'ipad', 'android','web')))
    AND dw_users.home_domain in ('us') and dw_users.is_valid_user is true
    AND (NOT coalesce((datediff((CASE WHEN dw_users.user_status = 'restricted' THEN dw_users.status_updated_at ELSE NULL END),
            coalesce(dw_users.guest_joined_at, dw_users.joined_at)) + 1) <= 30, FALSE))




In [0]:
%sql
select 
main_page_name, 
count(distinct event_id) as fm, 
count(distinct case when shopper_listing_first_booked_date = event_date then listing_id END)  AS fm_d1_oi,
count(distinct case when shopper_listing_first_order_id is not null then listing_id end)  AS fm_oi
from external_scratch.highway_traffic_enriched
where 1=1 
 and event_date between '2026-01-08' and '2025-01-12'
 and  actor_type = 'user'
 and shopper_listing_interaction_number = 1
 and shopper_user_domain = 'us' 
 and main_page_name in ('brand','search_results','aesthetic','cross_merch_listings','closet','look')
group by 1
order by 2 desc

QQ: Is there always a first Match happening for order_item? 

In [0]:
%sql 
select * from external_scratch.highway_traffic_enriched 
limit 5

In [0]:
### Attribute Look L0+L1 US only 

In [0]:
%sql
select 
  CASE 
    WHEN main_page_name in ('search_results', 'merch_search') THEN 'search_results' 
    ELSE main_page_name 
  END as main_page_name,  
  count(distinct event_id) as fm, 
  count(distinct case when shopper_listing_first_booked_date = event_date then listing_id END) as fm_d1_oi,
  count(distinct case when shopper_listing_first_order_id is not null then listing_id end) as fm_oi
from external_scratch.highway_traffic_enriched
where 1=1 
  and event_date between '2025-11-30' and '2025-12-13'
  and actor_type = 'user'
  and shopper_listing_interaction_number = 1
  and shopper_user_domain in ('us') 
  and main_page_name in (
    'brand',
    'search_results',
    'merch_search',
    'aesthetic',
    'cross_merch_listings',
    'closet',
    'look'
  )
group by 1
order by 2 desc

In [0]:
%sql
select 
  CASE 
    WHEN main_page_name in ('search_results', 'merch_search') THEN 'search_results' 
    ELSE main_page_name 
  END as main_page_name,  
  count(distinct event_id) as fm, 
  count(distinct case when shopper_listing_first_booked_date = event_date then listing_id END) as fm_d1_oi,
  count(distinct case when shopper_listing_first_order_id is not null then listing_id end) as fm_oi
from external_scratch.highway_traffic_enriched
where 1=1 
  and event_date between '2025-11-30' and '2025-12-13'
  and actor_type = 'user'
  and shopper_listing_interaction_number = 1
  and shopper_user_domain in ('us') 
  and main_page_name in (
    'brand',
    'search_results',
    'merch_search',
    'aesthetic',
    'cross_merch_listings',
    'closet',
    'look'
  )
group by 1
order by 2 desc

#### [Athena Report](https://docs.google.com/spreadsheets/d/1s507maHUuB9x-8u2eprvaGMxsbXy-lBPd3-5vx59o3Q/edit?pli=1&gid=1581439931#gid=1581439931)
week of 11/30 showing 1.3M order Items, why only 1.1here? 

In [0]:
%sql 
select count(distinct case when shopper_listing_first_order_id is not null then listing_id end) as fm_oi, 
COUNT(DISTINCT SHOPPER_LISTING_FIRST_ORDER_ID) AS fm_ordr_id
from external_scratch.highway_traffic_enriched
where 1=1 
 and event_date between '2025-11-30' and '2025-12-06'
 --and  actor_type = 'user'
 and shopper_listing_interaction_number = 1
 and shopper_user_domain in ('us','ca')  --- 1.3M IN TOTAL FOR ATHENA REPORT? 

In [0]:
### Asethetic page views 

In [0]:
%sql 
select 
home_domain, 
count(case when r.verb = 'view' AND r.direct_object.name = 'look' then r.actor.id else null end) as look_views,
count(case when r.verb = 'view' AND r.direct_object.name = 'search' and on.name = 'look'  then r.actor.id else null end) as look_search_views,
count(case when r.verb = 'view' AND r.direct_object.name = 'aesthetic' then r.actor.id else null end) as aesthetic_views
from raw_events r
inner join s3_analytics.dw_users dw_users on r.actor.id = dw_users.user_id

where  1=1
and event_date between '2026-01-09' and '2026-01-23'
and r.direct_object.name in ('aesthetic', 'look','search') and verb = 'view'
    AND r.actor.type IN ('user') 
    AND ((r.using.app_type IN ('iphone', 'ipad', 'android','web')))
    AND dw_users.home_domain in ('us') and dw_users.is_valid_user is true
    AND (NOT coalesce((datediff((CASE WHEN dw_users.user_status = 'restricted' THEN dw_users.status_updated_at ELSE NULL END),
            coalesce(dw_users.guest_joined_at, dw_users.joined_at)) + 1) <= 30, FALSE))
group by 1


The below file is generated in notebook [VisualDiscovery_02_ShoptheLook](https://poshmark-prod.cloud.databricks.com/editor/notebooks/4224641565907605?o=3891659053752709#command/6137012409913672)

In [0]:
bucket = "s3://analytics-scratch-reviewed-poshmark-production" 
team_name = "product_analytics"
project_name = "feed/visual_discovery/look"
data_name = "lookDF"

output_path = f"{bucket}/{team_name}/{project_name}/{data_name}/"
look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("ql_visual_discovery_look_page")

In [0]:
%sql 
select sum(look_clicks) from ql_visual_discovery_look_page
where home_domain = 'us' and location = 'footer' and verb = 'click'

In [0]:
%sql 
select sum(shopper_listing_first_gmv)
from external_scratch.highway_traffic_enriched
where 1=1 and (main_page_name = 'look'  or prev_page_name = 'look' )
 and event_date between '2025-12-07' and '2025-12-07'
 and  actor_type = 'user'
 and shopper_listing_interaction_number = 1 and shopper_listing_first_booked_date is not null 
 limit 5

##MainPage FM L0 Comparison and Page Views

SELECT
event_date, 
    COALESCE(SUM(a.net_d30_active_users ), 0) AS count_net_d30_active_users,
    COALESCE(SUM(a.first_match_order_items ), 0) AS count_first_match_order_items,
    COALESCE(SUM(a.search_first_match_order_items ), 0) AS count_search_first_match_order_items,
    COALESCE(SUM(a.community_closet_first_match_order_items ), 0) AS count_community_closet_first_match_order_items,
    COALESCE(SUM(a.show_first_match_order_items ), 0) AS count_show_first_match_order_items,
    COALESCE(SUM(a.brand_first_match_order_items ), 0) AS count_brand_first_match_order_items,
    COALESCE(SUM(a.web_paid_first_match_order_items ), 0) AS count_web_paid_first_match_order_items,
    COALESCE(SUM(a.cross_merch_first_match_order_items ), 0) AS count_cross_merch_first_match_order_items,
    COALESCE(SUM(a.feed_first_match_order_items ), 0) AS count_feed_first_match_order_items,
    COALESCE(SUM(a.all_other_first_match_order_items ), 0) AS count_all_other_first_match_order_items,
    COALESCE(SUM(a.total_first_match ), 0) AS count_first_matches,
    COALESCE(SUM(a.search_first_match ), 0) AS count_search_first_match,
    COALESCE(SUM(a.community_closet_first_match ), 0) count_community_closet_first_match,
    COALESCE(SUM(a.show_first_match ), 0) AS count_show_first_match,
    COALESCE(SUM(a.brand_first_match ), 0) AS count_brand_first_match,
    COALESCE(SUM(a.web_paid_first_match ), 0) AS count_web_paid_first_match,
    COALESCE(SUM(a.cross_merch_first_match ), 0) AS count_cross_merch_first_match,
    COALESCE(SUM(a.feed_first_match ), 0) AS count_feed_first_match,
    COALESCE(SUM(a.all_other_first_match ), 0) AS count_all_other_first_match,
    COALESCE(SUM(a.search_page_views ), 0) AS count_all_other_first_match,
    COALESCE(SUM(a.show_page_views ), 0) AS count_show_page_views,
    COALESCE(SUM(a.brand_page_views ), 0) AS count_brand_page_views,
    COALESCE(SUM(a.cross_merch_page_views ), 0) AS count_cross_merch_page_views,
    COALESCE(SUM(a.feed_page_views ), 0) AS count_feed_page_views,
    COALESCE(SUM(a.community_closet_page_views ), 0) AS count_community_closet_page_views,
    COALESCE(SUM(a.brand_first_match_order_item_gmv_usd ), 0) AS total_brand_first_match_order_item_gmv_usd,
    COALESCE(SUM(a.community_closet_first_match_order_item_gmv_usd ), 0) AS total_community_closet_first_match_order_item_gmv_usd,
    COALESCE(SUM(a.cross_merch_first_match_order_item_gmv_usd ), 0) AS total_cross_merch_first_match_order_item_gmv_usd,
    COALESCE(SUM(a.feed_first_match_order_item_gmv_usd ), 0) AS total_feed_first_match_order_item_gmv_usd,
    COALESCE(SUM(a.all_other_first_match_order_item_gmv_usd ), 0) AS total_all_other_first_match_order_item_gmv_usd,
    COALESCE(SUM(a.unattributed_first_match_order_item_gmv_usd ), 0) AS total_unattributed_first_match_order_item_gmv_usd,
    COALESCE(SUM(a.search_first_match_order_item_gmv_usd ), 0) AS total_search_first_match_order_item_gmv_usd,
    COALESCE(SUM(a.show_first_match_order_item_gmv_usd ), 0) AS total_show_first_match_order_item_gmv_usd,
    COALESCE(SUM(a.web_paid_first_match_order_item_gmv_usd ), 0) AS total_web_paid_first_match_order_item_gmv_usd,
    COALESCE(SUM(a.unattributed_first_match ), 0) AS count_unattributed_first_match,
    COALESCE(SUM(a.unattributed_first_match_order_items ), 0) AS count_unattributed_first_match_order_items
FROM external_scratch.athena_matching_engine_daily  a
WHERE (a.event_date ) between '2025-12-01' and '2025-12-01'
GROUP BY
    1
ORDER BY
    1 DESC
